# Notebook 18 — Observability, Metrics, Tracing, and Cost Accounting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/18_observability_metrics_tracing_and_cost_accounting.ipynb)

Module 5 starts where pure benchmarking stops. Earlier notebooks taught how to measure latency, throughput, memory, routing, and overload behavior. This notebook turns those measurements into operational telemetry you can use during live serving, post-release review, and cost-aware capacity planning.

## What you will build

- a notebook-local telemetry stream for a small LLM serving system
- route, tenant, and endpoint latency and throughput dashboards
- lightweight request tracing with critical-path inspection
- request accounting and cost-proxy reports for chargeback decisions
- an eval-linked scorecard that reconnects operations work to the earlier `evals/` track

## Reconnecting to `evals/`

The `evals/` track taught you to score quality, structured-output correctness, robustness, and release regressions before deployment. Operations extends that same discipline into production telemetry:

- evals ask whether answers are good enough
- observability asks whether good answers arrive fast enough, reliably enough, and cheaply enough
- tracing asks where the latency budget was spent
- accounting asks who consumed the budget and whether the release is still viable

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add notebook helpers and artifact paths

We will keep the notebook transparent: plain Python data structures, pandas for summaries, and small helper functions for markdown tables and JSON artifacts.

In [ ]:
random.seed(18)

ARTIFACT_DIR = ARTIFACT_ROOT / "notebook_18_observability"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def percentile(values, pct):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return 0.0
    rank = (len(ordered) - 1) * (pct / 100)
    lower = math.floor(rank)
    upper = math.ceil(rank)
    if lower == upper:
        return ordered[int(rank)]
    weight = rank - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

def sparkbar(value, maximum, width=18):
    if maximum <= 0:
        ratio = 0.0
    else:
        ratio = max(0.0, min(1.0, value / maximum))
    used = int(round(ratio * width))
    return "█" * used + "·" * (width - used)

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define a notebook-local service and telemetry schema

The telemetry stream models a small serving stack with three runtime routes and several tenants. Each request records queue time, prefill time, decode time, total latency, token counts, cache behavior, and a trace id.

In [ ]:
service_routes = {
    "chat-balanced": {
        "base_prompt_tokens": 620,
        "base_output_tokens": 180,
        "base_queue_ms": 42,
        "base_prefill_ms": 280,
        "base_decode_ms": 820,
        "error_bias": 0.022,
        "quality_hint": 0.88,
    },
    "extract-fast": {
        "base_prompt_tokens": 340,
        "base_output_tokens": 90,
        "base_queue_ms": 18,
        "base_prefill_ms": 150,
        "base_decode_ms": 360,
        "error_bias": 0.013,
        "quality_hint": 0.82,
    },
    "safety-fallback": {
        "base_prompt_tokens": 540,
        "base_output_tokens": 130,
        "base_queue_ms": 55,
        "base_prefill_ms": 240,
        "base_decode_ms": 520,
        "error_bias": 0.018,
        "quality_hint": 0.93,
    },
}

tenants = [
    {"tenant": "castalia-mentor", "mix": {"chat-balanced": 0.65, "extract-fast": 0.25, "safety-fallback": 0.10}},
    {"tenant": "research-lab", "mix": {"chat-balanced": 0.45, "extract-fast": 0.40, "safety-fallback": 0.15}},
    {"tenant": "ops-console", "mix": {"chat-balanced": 0.30, "extract-fast": 0.20, "safety-fallback": 0.50}},
]

endpoint_for_route = {
    "chat-balanced": "/v1/chat/completions",
    "extract-fast": "/v1/responses",
    "safety-fallback": "/v1/chat/completions",
}

def weighted_pick(weight_map, rng):
    roll = rng.random()
    cursor = 0.0
    items = list(weight_map.items())
    for name, weight in items:
        cursor += weight
        if roll <= cursor:
            return name
    return items[-1][0]

def build_request_record(index):
    rng = random.Random(f"req-{index}")
    tenant_profile = tenants[index % len(tenants)]
    route = weighted_pick(tenant_profile["mix"], rng)
    route_profile = service_routes[route]
    prompt_tokens = int(rng.gauss(route_profile["base_prompt_tokens"], route_profile["base_prompt_tokens"] * 0.14))
    prompt_tokens = max(prompt_tokens, 80)
    output_tokens = int(rng.gauss(route_profile["base_output_tokens"], route_profile["base_output_tokens"] * 0.18))
    output_tokens = max(output_tokens, 24)
    cache_hit = rng.random() < (0.24 if route == "chat-balanced" else 0.18)
    queue_ms = max(2.0, rng.gauss(route_profile["base_queue_ms"], route_profile["base_queue_ms"] * 0.30))
    prefill_multiplier = 0.72 if cache_hit else 1.0
    prefill_ms = max(35.0, rng.gauss(route_profile["base_prefill_ms"] * prefill_multiplier, route_profile["base_prefill_ms"] * 0.12))
    decode_ms = max(40.0, rng.gauss(route_profile["base_decode_ms"], route_profile["base_decode_ms"] * 0.18))
    postprocess_ms = max(4.0, rng.gauss(18.0, 5.0))
    total_latency_ms = queue_ms + prefill_ms + decode_ms + postprocess_ms
    ttft_ms = queue_ms + prefill_ms + max(12.0, decode_ms * 0.09)
    input_tokens = prompt_tokens
    billed_output_tokens = output_tokens
    token_rate_tps = billed_output_tokens / max(0.05, decode_ms / 1000)
    status_roll = rng.random()
    if status_roll < route_profile["error_bias"] * 0.35:
        status_code = 429
    elif status_roll < route_profile["error_bias"]:
        status_code = 500
    else:
        status_code = 200
    success = status_code == 200
    timestamp = pd.Timestamp("2026-05-18 09:00:00") + pd.Timedelta(minutes=index * 2)
    return {
        "request_id": f"req_{index:04d}",
        "trace_id": f"trace_{index:04d}",
        "tenant": tenant_profile["tenant"],
        "route": route,
        "endpoint": endpoint_for_route[route],
        "timestamp": timestamp,
        "status_code": status_code,
        "success": success,
        "cache_hit": cache_hit,
        "input_tokens": input_tokens,
        "output_tokens": billed_output_tokens,
        "queue_ms": round(queue_ms, 2),
        "prefill_ms": round(prefill_ms, 2),
        "decode_ms": round(decode_ms, 2),
        "ttft_ms": round(ttft_ms, 2),
        "duration_ms": round(total_latency_ms, 2),
        "token_rate_tps": round(token_rate_tps, 2),
        "quality_hint": route_profile["quality_hint"],
    }

request_records = [build_request_record(index) for index in range(240)]
print("Requests:", len(request_records))
print("Routes:", sorted({row["route"] for row in request_records}))

## Step 3: Materialize the request log

A real service would ship this to a metrics backend and a trace collector. In the notebook we keep the same shape, but we write JSONL and CSV artifacts so you can inspect every row.

In [ ]:
requests_df = pd.DataFrame(request_records).sort_values("timestamp").reset_index(drop=True)
requests_jsonl = ARTIFACT_DIR / "requests.jsonl"
requests_jsonl.write_text("\n".join(json.dumps(row, default=str) for row in request_records), encoding="utf-8")
requests_df.to_csv(ARTIFACT_DIR / "requests.csv", index=False)

preview_columns = [
    "request_id",
    "tenant",
    "route",
    "status_code",
    "cache_hit",
    "input_tokens",
    "output_tokens",
    "ttft_ms",
    "duration_ms",
    "token_rate_tps",
]
print(requests_df[preview_columns].head(8).to_markdown(index=False))
print("Saved:", requests_jsonl.name)

## Step 4: Derive route-level latency and throughput metrics

Operational review starts with clear aggregates: request volume, success rate, latency percentiles, and output token rate. Percentiles matter because tail latency, not mean latency, decides whether the system feels healthy.

In [ ]:
route_rows = []
for route, group in requests_df.groupby("route"):
    route_rows.append({
        "route": route,
        "requests": int(len(group)),
        "success_rate": round(group["success"].mean(), 4),
        "p50_latency_ms": round(percentile(group["duration_ms"], 50), 2),
        "p95_latency_ms": round(percentile(group["duration_ms"], 95), 2),
        "p99_latency_ms": round(percentile(group["duration_ms"], 99), 2),
        "p95_ttft_ms": round(percentile(group["ttft_ms"], 95), 2),
        "avg_token_rate_tps": round(group["token_rate_tps"].mean(), 2),
        "cache_hit_rate": round(group["cache_hit"].mean(), 4),
        "total_output_tokens": int(group["output_tokens"].sum()),
    })

route_metrics = pd.DataFrame(route_rows).sort_values("p95_latency_ms")
route_metrics.to_csv(ARTIFACT_DIR / "route_metrics.csv", index=False)
print(route_metrics.to_markdown(index=False))

## Step 5: Visualize latency breakdown by queue, prefill, and decode

A single p95 number hides the cause of slowness. Breaking latency into queue, prefill, and decode is the first step toward deciding whether to optimize batching, cache reuse, or downstream post-processing.

In [ ]:
latency_breakdown = (
    requests_df.groupby("route")[["queue_ms", "prefill_ms", "decode_ms"]]
    .mean()
    .round(2)
    .reset_index()
)
latency_breakdown.to_csv(ARTIFACT_DIR / "latency_breakdown.csv", index=False)

plot_df = latency_breakdown.set_index("route")
ax = plot_df.plot(kind="bar", stacked=True, figsize=(9, 4), color=["#4c78a8", "#f58518", "#54a24b"])
ax.set_title("Average latency budget by route")
ax.set_ylabel("milliseconds")
ax.set_xlabel("route")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "latency_breakdown.png", dpi=160)
plt.show()
print(latency_breakdown.to_markdown(index=False))

## Step 6: Add service-level indicators and saturation proxies

Metrics should also answer whether the system is trending toward overload. We will compute 20-minute windows with queue pressure, output-token throughput, and error rate proxies.

In [ ]:
windowed = (
    requests_df.set_index("timestamp")
    .groupby([pd.Grouper(freq="20min"), "route"])
    .agg(
        requests=("request_id", "count"),
        error_rate=("success", lambda values: round(1 - float(np.mean(values)), 4)),
        avg_queue_ms=("queue_ms", "mean"),
        p95_latency_ms=("duration_ms", lambda values: round(percentile(values, 95), 2)),
        total_output_tokens=("output_tokens", "sum"),
        avg_token_rate_tps=("token_rate_tps", "mean"),
    )
    .reset_index()
)
windowed["throughput_tokens_per_min"] = (windowed["total_output_tokens"] / 20).round(2)
windowed["queue_pressure"] = (windowed["avg_queue_ms"] / windowed["avg_queue_ms"].max()).round(3)
windowed["slo_breach"] = (windowed["p95_latency_ms"] > 1450) | (windowed["error_rate"] > 0.03)
windowed.to_csv(ARTIFACT_DIR / "windowed_sli.csv", index=False)

sli_preview = windowed[["timestamp", "route", "requests", "error_rate", "p95_latency_ms", "throughput_tokens_per_min", "queue_pressure", "slo_breach"]].head(12)
print(sli_preview.to_markdown(index=False))
print("Breached windows:", int(windowed["slo_breach"].sum()))

## Step 7: Build a lightweight tracing model

Metrics tell you that the system is slow. Traces tell you **where** it became slow. We will create notebook-local spans for gateway, routing, prefill, decode, and response serialization.

In [ ]:
span_records = []
for row in requests_df.to_dict(orient="records"):
    cursor = 0.0
    spans = [
        ("gateway", None, 0.0, 4.0),
        ("router", "gateway", 4.0, 14.0),
        ("queue_wait", "router", 14.0, 14.0 + row["queue_ms"]),
        ("prefill", "queue_wait", 14.0 + row["queue_ms"], 14.0 + row["queue_ms"] + row["prefill_ms"]),
        ("decode", "prefill", 14.0 + row["queue_ms"] + row["prefill_ms"], 14.0 + row["queue_ms"] + row["prefill_ms"] + row["decode_ms"]),
        ("postprocess", "decode", 14.0 + row["queue_ms"] + row["prefill_ms"] + row["decode_ms"], row["duration_ms"]),
    ]
    for name, parent, start_ms, end_ms in spans:
        span_records.append({
            "trace_id": row["trace_id"],
            "request_id": row["request_id"],
            "route": row["route"],
            "tenant": row["tenant"],
            "span_name": name,
            "parent_span": parent,
            "start_ms": round(start_ms, 2),
            "end_ms": round(end_ms, 2),
            "duration_ms": round(end_ms - start_ms, 2),
        })

spans_df = pd.DataFrame(span_records)
spans_df.to_csv(ARTIFACT_DIR / "trace_spans.csv", index=False)
print(spans_df.head(12).to_markdown(index=False))

## Step 8: Inspect one critical trace

When an alert fires, you rarely inspect all traces. You inspect a representative slow trace, compare the major spans, and decide which subsystem owns the latency budget.

In [ ]:
def render_trace(trace_frame):
    lines_out = []
    for span in trace_frame.sort_values(["start_ms", "duration_ms"]).to_dict(orient="records"):
        parent = span["parent_span"] or "root"
        lines_out.append(f'{span["span_name"]:>12} | parent={parent:<10} | {span["duration_ms"]:>7.2f} ms')
    return "\n".join(lines_out)

slowest_request = requests_df.sort_values("duration_ms", ascending=False).iloc[0]
selected_trace = spans_df[spans_df["trace_id"] == slowest_request["trace_id"]].copy()
critical_path_ms = round(selected_trace["duration_ms"].sum(), 2)
trace_summary = {
    "trace_id": slowest_request["trace_id"],
    "request_id": slowest_request["request_id"],
    "route": slowest_request["route"],
    "tenant": slowest_request["tenant"],
    "duration_ms": float(slowest_request["duration_ms"]),
    "critical_path_ms": critical_path_ms,
    "span_rows": selected_trace.to_dict(orient="records"),
}
write_json(ARTIFACT_DIR / "selected_trace.json", trace_summary)
print("Selected trace:", slowest_request["trace_id"])
print(render_trace(selected_trace))

## Step 9: Build request accounting by tenant and route

Accounting is the operational bridge from telemetry to budget. We are not using provider invoices here; instead, we compute billable units from tokens, errors, and cache misses so the notebook remains portable.

In [ ]:
requests_df["cache_miss"] = (~requests_df["cache_hit"]).astype(int)
tenant_accounting = (
    requests_df.groupby(["tenant", "route"])
    .agg(
        requests=("request_id", "count"),
        success_rate=("success", "mean"),
        input_tokens=("input_tokens", "sum"),
        output_tokens=("output_tokens", "sum"),
        cache_misses=("cache_miss", "sum"),
        total_latency_ms=("duration_ms", "sum"),
    )
    .reset_index()
)
tenant_accounting["success_rate"] = tenant_accounting["success_rate"].round(4)
tenant_accounting["avg_latency_ms"] = (tenant_accounting["total_latency_ms"] / tenant_accounting["requests"]).round(2)
tenant_accounting.to_csv(ARTIFACT_DIR / "tenant_accounting.csv", index=False)
print(tenant_accounting.to_markdown(index=False))

## Step 10: Estimate cost proxies from runtime behavior

Portable cost accounting often starts with proxies instead of invoices. We will assign cost weights to prompt tokens, output tokens, runtime milliseconds, and trace storage so teams can compare routes before a formal billing integration exists.

In [ ]:
cost_weights = {
    "input_token": 0.0000018,
    "output_token": 0.0000031,
    "latency_ms": 0.00000022,
    "cache_miss": 0.00042,
    "trace_span": 0.00003,
}

span_count_by_trace = spans_df.groupby("trace_id").size().rename("span_count")
cost_df = requests_df.merge(span_count_by_trace, left_on="trace_id", right_index=True)
cost_df["estimated_cost_usd"] = (
    cost_df["input_tokens"] * cost_weights["input_token"]
    + cost_df["output_tokens"] * cost_weights["output_token"]
    + cost_df["duration_ms"] * cost_weights["latency_ms"]
    + cost_df["cache_miss"] * cost_weights["cache_miss"]
    + cost_df["span_count"] * cost_weights["trace_span"]
)
cost_df["estimated_cost_usd"] = cost_df["estimated_cost_usd"].round(5)

route_costs = (
    cost_df.groupby("route")
    .agg(
        requests=("request_id", "count"),
        total_estimated_cost_usd=("estimated_cost_usd", "sum"),
        avg_estimated_cost_usd=("estimated_cost_usd", "mean"),
        avg_latency_ms=("duration_ms", "mean"),
    )
    .reset_index()
)
route_costs["total_estimated_cost_usd"] = route_costs["total_estimated_cost_usd"].round(4)
route_costs["avg_estimated_cost_usd"] = route_costs["avg_estimated_cost_usd"].round(5)
route_costs["avg_latency_ms"] = route_costs["avg_latency_ms"].round(2)
route_costs.to_csv(ARTIFACT_DIR / "route_costs.csv", index=False)
print(route_costs.to_markdown(index=False))

## Step 11: Rejoin operations telemetry with eval release criteria

This is the explicit handoff back to `evals/`: do not look at cost or latency in isolation. Merge operational data with quality, schema, and robustness pass rates so release decisions remain multi-objective.

In [ ]:
eval_scorecard = pd.DataFrame([
    {"route": "chat-balanced", "rubric_score": 0.86, "schema_pass_rate": 0.97, "robustness_pass_rate": 0.92},
    {"route": "extract-fast", "rubric_score": 0.79, "schema_pass_rate": 0.95, "robustness_pass_rate": 0.89},
    {"route": "safety-fallback", "rubric_score": 0.91, "schema_pass_rate": 0.99, "robustness_pass_rate": 0.96},
])

ops_eval_scorecard = route_metrics.merge(route_costs, on=["route", "requests"], how="left").merge(eval_scorecard, on="route", how="left")
ops_eval_scorecard["gate_pass"] = (
    (ops_eval_scorecard["rubric_score"] >= 0.80)
    & (ops_eval_scorecard["schema_pass_rate"] >= 0.96)
    & (ops_eval_scorecard["robustness_pass_rate"] >= 0.90)
    & (ops_eval_scorecard["p95_latency_ms"] <= 1450)
    & (ops_eval_scorecard["success_rate"] >= 0.97)
    & (ops_eval_scorecard["avg_estimated_cost_usd"] <= 0.0085)
 )
ops_eval_scorecard.to_csv(ARTIFACT_DIR / "ops_eval_scorecard.csv", index=False)
print(ops_eval_scorecard[["route", "rubric_score", "schema_pass_rate", "robustness_pass_rate", "p95_latency_ms", "success_rate", "avg_estimated_cost_usd", "gate_pass"]].to_markdown(index=False))

## Step 12: Publish a practical daily report artifact

Notebook work becomes operationally useful when it leaves behind artifacts a reviewer can consume: CSV summaries, JSON scorecards, and a compact markdown report.

In [ ]:
max_cost = route_costs["total_estimated_cost_usd"].max()
route_report_rows = []
for row in route_costs.sort_values("total_estimated_cost_usd", ascending=False).to_dict(orient="records"):
    route_report_rows.append({
        "route": row["route"],
        "requests": row["requests"],
        "cost_bar": sparkbar(row["total_estimated_cost_usd"], max_cost),
        "total_estimated_cost_usd": row["total_estimated_cost_usd"],
    })

daily_report = {
    "module": "systems-05",
    "notebook": 18,
    "summary": {
        "total_requests": int(len(requests_df)),
        "global_success_rate": round(float(requests_df["success"].mean()), 4),
        "global_p95_latency_ms": round(percentile(requests_df["duration_ms"], 95), 2),
        "estimated_total_cost_usd": round(float(cost_df["estimated_cost_usd"].sum()), 4),
    },
    "artifacts": {
        "requests": "requests.csv",
        "route_metrics": "route_metrics.csv",
        "windowed_sli": "windowed_sli.csv",
        "trace_spans": "trace_spans.csv",
        "scorecard": "ops_eval_scorecard.csv",
    },
}
write_json(ARTIFACT_DIR / "daily_report.json", daily_report)

report_md = "\n".join([
    "# Module 5 Observability Report",
    "",
    "## Route cost heat",
    to_markdown_table(route_report_rows, ["route", "requests", "cost_bar", "total_estimated_cost_usd"]),
    "",
    "## Release gate summary",
    ops_eval_scorecard[["route", "gate_pass"]].to_markdown(index=False),
])
(ARTIFACT_DIR / "daily_report.md").write_text(report_md, encoding="utf-8")
print(report_md)

## Recap

You now have the operational side of the same discipline introduced in `evals/`: observable latency, throughput, tracing, request accounting, and cost-aware release gates. The next notebook uses these artifacts to drive canaries, rollback rules, and incident playbooks.